# 🎙️ TTS desde cero — Entrenamiento en Colab

Ejecuta las celdas en orden.
Si la sesión se desconecta, vuelve a ejecutar desde la celda 1.
El entrenamiento continuará desde el último checkpoint guardado en Drive.

---
## Celda 1 — Verificar GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No hay GPU disponible. Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → A100')

print(f'GPU:   {torch.cuda.get_device_name(0)}')
print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA:  {torch.version.cuda}')
print('✅ GPU lista')

---
## Celda 2 — Montar Google Drive

Los checkpoints se guardan aquí.
Si la sesión se desconecta, el progreso no se pierde.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/tts_checkpoints', exist_ok=True)
print('✅ Google Drive montado')
print('📁 Checkpoints en: /content/drive/MyDrive/tts_checkpoints')

---
## Celda 3 — Clonar repositorio

Cambia la URL por la de tu repositorio.

In [ ]:
import os

REPO_URL = 'https://github.com/TU_USUARIO/tts_project.git'  # ← cambia esto

if os.path.exists('/content/tts_project'):
    # si ya existe el repo solo actualizamos
    %cd /content/tts_project
    !git pull
    print('✅ Repositorio actualizado')
else:
    # primera vez — clonar
    %cd /content
    !git clone {REPO_URL}
    %cd tts_project
    print('✅ Repositorio clonado')

!echo "Directorio actual: $(pwd)"
!ls

---
## Celda 4 — Instalar dependencias

In [ ]:
%cd /content/tts_project

!pip install uv -q
!uv sync

# verificar que los imports funcionan
!uv run python -c "import torch, librosa, soundfile; print('✅ Dependencias instaladas')"

---
## Celda 5 — Descargar dataset CSS10 Spanish

Solo necesitas ejecutar esta celda la primera vez.
Si el dataset ya está descargado puedes saltarla.

Necesitas tu `kaggle.json` — descárgalo desde:
kaggle.com → tu perfil → Settings → API → Create New Token

In [ ]:
import os

# comprobar si el dataset ya existe
if os.path.exists('/content/tts_project/data/css10_es/transcript.txt'):
    print('✅ Dataset ya descargado')
else:
    print('Descargando dataset...')

    # subir kaggle.json
    from google.colab import files
    print('Selecciona tu kaggle.json:')
    files.upload()

    # configurar credenciales
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json

    # descargar y descomprimir
    os.makedirs('/content/tts_project/data/css10_es', exist_ok=True)
    !uv run kaggle datasets download \
        -d bryanpark/spanish-single-speaker-speech-dataset \
        -p /content/tts_project/data

    !unzip -q /content/tts_project/data/spanish-single-speaker-speech-dataset.zip \
        -d /content/tts_project/data/css10_es

    print('✅ Dataset descargado')

# verificar estructura
!ls /content/tts_project/data/css10_es
!wc -l /content/tts_project/data/css10_es/transcript.txt

---
## Celda 6 — Verificar pipeline completo

Antes de entrenar comprobamos que todo el pipeline funciona.

In [ ]:
%cd /content/tts_project

!uv run python -c "
import torch
from src.data.text       import VOCAB_SIZE, texto_a_indices
from src.data.dataset    import TTSDataset, crear_dataloader
from src.pipeline.encoder   import Encoder
from src.pipeline.attention import Attention
from src.pipeline.decoder   import Decoder

device = torch.device('cuda')

# modelos pequeños para la prueba
encoder   = Encoder(VOCAB_SIZE, 128, 128).to(device)
attention = Attention(128).to(device)
decoder   = Decoder(128, 80).to(device)

# prueba con una frase
frase   = 'hola, ¿en qué te puedo ayudar?'
indices = torch.tensor(texto_a_indices(frase)).unsqueeze(0).to(device)

with torch.no_grad():
    enc_salida = encoder(indices)
    mel        = decoder(enc_salida, attention)

print(f'Encoder salida: {enc_salida.shape}')
print(f'Mel generado:   {mel.shape}')

# prueba dataset
dataset = TTSDataset('data/css10_es')
print(f'Dataset: {len(dataset)} muestras')
print('✅ Pipeline completo funcionando')
"

---
## Celda 7 — Configuración del entrenamiento

Ajusta los hiperparámetros aquí antes de entrenar.

In [ ]:
# ── Hiperparámetros ──────────────────────────────────────────
# puedes modificar estos valores antes de entrenar

HIDDEN_DIM    = 512     # tamaño del estado interno de encoder y decoder
EMBEDDING_DIM = 512     # tamaño de cada embedding de carácter
N_MELS        = 80      # frecuencias del Mel Spectrogram
BATCH_SIZE    = 32      # frases por batch — A100 aguanta 32 sin problema
EPOCHS        = 200     # épocas totales de entrenamiento
LR            = 0.001   # learning rate

# ruta del checkpoint en Google Drive
CHECKPOINT = '/content/drive/MyDrive/tts_checkpoints/ultimo.pt'

# directorio del dataset
DATA_DIR = '/content/tts_project/data/css10_es'

print('Configuración:')
print(f'  HIDDEN_DIM:    {HIDDEN_DIM}')
print(f'  EMBEDDING_DIM: {EMBEDDING_DIM}')
print(f'  BATCH_SIZE:    {BATCH_SIZE}')
print(f'  EPOCHS:        {EPOCHS}')
print(f'  LR:            {LR}')
print(f'  CHECKPOINT:    {CHECKPOINT}')

---
## Celda 8 — Arrancar entrenamiento

Esta celda puede tardar horas o días.
Los checkpoints se guardan cada 10 épocas en Google Drive.
Si la sesión se desconecta vuelve a ejecutar desde la celda 1
y el entrenamiento continuará desde donde lo dejó.

In [ ]:
%cd /content/tts_project

# escribir la configuración en un fichero temporal
# para que train.py la lea
config = f"""
HIDDEN_DIM    = {HIDDEN_DIM}
EMBEDDING_DIM = {EMBEDDING_DIM}
N_MELS        = {N_MELS}
BATCH_SIZE    = {BATCH_SIZE}
EPOCHS        = {EPOCHS}
LR            = {LR}
CHECKPOINT    = '{CHECKPOINT}'
DATA_DIR      = '{DATA_DIR}'
"""

with open('config.py', 'w') as f:
    f.write(config)

print('Iniciando entrenamiento...')
print('Los checkpoints se guardan cada 10 épocas en Google Drive')
print('='*50)

!uv run python train.py

---
## Celda 9 — Escuchar resultados con Griffin-Lim

Genera audio desde texto usando el modelo entrenado.
Usa Griffin-Lim para reconstruir el audio desde el Mel Spectrogram.
No necesita HiFi-GAN — útil para escuchar el progreso.

In [ ]:
%cd /content/tts_project

import torch
import soundfile as sf
from IPython.display import Audio, display

from src.data.text       import VOCAB_SIZE, texto_a_indices
from src.data.audio      import mel_a_wav, SAMPLE_RATE
from src.pipeline.encoder   import Encoder
from src.pipeline.attention import Attention
from src.pipeline.decoder   import Decoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# cargar modelos
encoder   = Encoder(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM).to(device)
attention = Attention(HIDDEN_DIM).to(device)
decoder   = Decoder(HIDDEN_DIM, N_MELS).to(device)

# cargar checkpoint
checkpoint = torch.load(CHECKPOINT, map_location=device)
encoder.load_state_dict(checkpoint['encoder'])
attention.load_state_dict(checkpoint['attention'])
decoder.load_state_dict(checkpoint['decoder'])

encoder.eval()
attention.eval()
decoder.eval()

# ── generar audio ────────────────────────────────────────────
FRASE = 'hola, ¿en qué te puedo ayudar?'   # ← cambia esto

indices = torch.tensor(texto_a_indices(FRASE)).unsqueeze(0).to(device)

with torch.no_grad():
    enc_salida = encoder(indices)
    mel        = decoder(enc_salida, attention)

# mel tiene shape (1, T, 80) — lo transponemos a (80, T)
mel_np = mel[0].T.cpu().numpy()

# reconstruir audio con Griffin-Lim
audio = mel_a_wav(mel_np)

# guardar y reproducir
sf.write('resultado.wav', audio, SAMPLE_RATE)
print(f'Frase: "{FRASE}"')
print(f'Frames generados: {mel.shape[1]}')
print(f'Duración: {len(audio)/SAMPLE_RATE:.2f} segundos')
display(Audio(audio, rate=SAMPLE_RATE))

---
## Celda 10 — Actualizar código desde GitHub

Si modificas el código en local y haces push,
ejecuta esta celda para que Colab use la versión actualizada.

In [ ]:
%cd /content/tts_project
!git pull
print('✅ Código actualizado')

---
## Celda 11 — Ver checkpoints guardados en Drive

In [ ]:
import os

carpeta = '/content/drive/MyDrive/tts_checkpoints'
ficheros = os.listdir(carpeta)

if not ficheros:
    print('No hay checkpoints todavía')
else:
    print(f'Checkpoints en {carpeta}:')
    for f in sorted(ficheros):
        ruta  = os.path.join(carpeta, f)
        tamaño = os.path.getsize(ruta) / 1e6
        print(f'  {f}  ({tamaño:.1f} MB)')